In [1]:
!pip install scikit-learn
!pip install transformers==4.38.2 datasets==2.16.1 tokenizers==0.15.2 accelerate==0.27.2 lime evaluate openpyxl
!pip uninstall -y numpy
!pip install numpy==1.26.4

import torch; 
print(torch.__version__); 
print(torch.version.cuda); 
print(torch.cuda.is_available());
print(torch.cuda.get_device_name(0))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 146.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 241.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.9/16.9 MB 252.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.3
    Uninstalling numpy-1.26.3:
      Successfully uninstalled numpy-1.26.3

[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python -m pip install --upgrade pip
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 190.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 150.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 204.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 87.7 MB

In [2]:

import torch; print(torch.__version__); print(torch.version.cuda); print(torch.cuda.is_available()); print(torch.cuda.get_device_name(0))

2.4.1+cu124
12.4
True
NVIDIA RTX A5000


In [9]:
from sklearn.metrics import classification_report, f1_score, confusion_matrix, precision_recall_curve, auc
from transformers import AutoTokenizer, AutoModel, AutoConfig, Trainer, TrainingArguments, PreTrainedModel
from transformers.modeling_outputs import SequenceClassifierOutput
from transformers import RobertaTokenizer, RobertaForSequenceClassification
from transformers import AutoModelForSequenceClassification, BertModel, BertConfig,BertPreTrainedModel
from lime.lime_text import LimeTextExplainer
from sklearn.utils.class_weight import compute_class_weight
import torch, torch.nn as nn
from datasets import Dataset
import evaluate, gc, time, numpy as np, pandas as pd, datetime, pickle, csv, re, platform, os, sys
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from transformers import  AutoModelForSequenceClassification, TrainerCallback, PretrainedConfig
import matplotlib.pyplot as plt
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def parse_vector_column(vec):
    if pd.isna(vec):return []
    if isinstance(vec, str):
        vec = vec.strip()
        if not vec:return []
        return [float(x) for x in vec.split()]
    return [float(x) for x in vec]

In [4]:
MAX_LENGTH = 128
modelname = "mBERT"
num_labels = 5
modelpath = "./models/"
N_EPOCHS = 5
lamda = 0

file_path = "/workspace/Data/UOLDRationales-24k-V2.xlsx"
result_path =  "/workspace/Results/T2-M1-Result.csv"
LOCAL_OUT = f"/workspace/lc_models/{modelname}" 
save_dir = "/workspace/ft_models/t2-m1-mBERT"
MODEL_NAME = "/workspace/models/bert-base-multilingual-cased"

df = pd.read_excel(file_path, engine='openpyxl')
df.reset_index( drop=True, inplace=True )
df.Tweet = df.Tweet.astype( 'str' )
print(df.columns, df.shape)
xcolumn = "Tweet"
ycolumn = "Tag"
rationale_column = "Vector"

print(df[ycolumn].value_counts().sort_index())
print(datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
df = df[[xcolumn, ycolumn, rationale_column]].dropna()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gc.collect()
torch.cuda.empty_cache()

Index(['Unnamed: 0.1', 'Unnamed: 0', 'TweetId', 'Tweet', 'Vector', 'Words',
       'TweetText', 'TweetSpaceInserted', 'Tag'],
      dtype='str') (24063, 9)
Tag
0    15404
1     4823
2     1168
3     2105
4      563
Name: count, dtype: int64
2026-02-28 10:08:11


In [5]:
gc.collect()
torch.cuda.empty_cache()

df['rationale'] = df[rationale_column].apply(parse_vector_column)

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df[ycolumn], random_state=42)
train_df, val_df = train_test_split(train_df, test_size=0.1, stratify=train_df[ycolumn],random_state=42)

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)
test_dataset = Dataset.from_pandas(test_df)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(examples[xcolumn],
     truncation=True,padding="max_length",
     max_length=MAX_LENGTH,return_attention_mask=True,)   
    
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset = train_dataset.rename_column(ycolumn, "labels")
val_dataset = val_dataset.rename_column(ycolumn, "labels")
test_dataset = test_dataset.rename_column(ycolumn, "labels")

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
val_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/17325 [00:00<?, ? examples/s]

Map:   0%|          | 0/1925 [00:00<?, ? examples/s]

Map:   0%|          | 0/4813 [00:00<?, ? examples/s]

In [11]:
accuracy_metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):logits = logits[0]
    preds = np.argmax(logits, axis=-1) 
    return { "accuracy": accuracy_metric.compute(predictions=preds, references=labels)["accuracy"],
        "f1_macro": f1_score(labels, preds, average="macro", zero_division=0),
        "f1_micro": f1_score(labels, preds, average="micro", zero_division=0),
        "f1_weighted": f1_score(labels, preds, average="weighted", zero_division=0),}

class MBertMulticlass(BertPreTrainedModel):
    config_class = BertConfig
    base_model_prefix = "bert"
    _supports_experts = False
    all_tied_weights_keys = {}
    def __init__(self, config):
        super().__init__(config)
        self.bert = BertModel(config)
        self.dropout = nn.Dropout(config.hidden_dropout_prob)
        self.classifier = nn.Linear(config.hidden_size, config.num_labels)
        self.classifier.weight.data.normal_(mean=0.0, std=0.02)
        self.classifier.bias.data.zero_()
        self.register_buffer("pos_weight", torch.tensor(config.num_labels, dtype=torch.float32))
        self.post_init()
    def forward(self, input_ids=None, attention_mask=None, labels=None, output_attentions=None, **kwargs):
        if output_attentions is None:
            output_attentions = self.config.output_attentions
        outputs = self.bert(input_ids=input_ids,attention_mask=attention_mask,
            output_attentions=output_attentions,)
        
        pooled_output = outputs.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output) 
        logits = torch.clamp(logits, -20.0, 20.0)
        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.pos_weight)
            loss = loss_fct(logits, labels.long())
        attentions = outputs.attentions if output_attentions else None
        
        return SequenceClassifierOutput(loss=loss,logits=logits,attentions=attentions,)



train_labels_array = np.array(train_dataset["labels"])
class_weights = compute_class_weight(
    class_weight='balanced',classes=np.unique(train_labels_array),
    y=train_labels_array )
pos_weight_cpu = torch.tensor(class_weights, dtype=torch.float)
print("Class weights:", pos_weight_cpu)

config = BertConfig.from_pretrained(MODEL_NAME)
config.num_labels = 5
config.return_dict = True
config.output_attentions = False 
config.use_cache = False
model = MBertMulticlass.from_pretrained(MODEL_NAME, config=config, use_safetensors=True, ignore_mismatched_sizes=True)
model.config.output_attentions = False
model.config.use_cache = False
model.register_buffer("pos_weight", pos_weight_cpu)
model.to(device)

training_args = TrainingArguments(
    output_dir=LOCAL_OUT,
    num_train_epochs=N_EPOCHS,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    evaluation_strategy="epoch", logging_steps=100,
    save_strategy="epoch", save_steps=100,
    save_total_limit=2, 
    fp16=False,
    fp16_full_eval=False,
    bf16=True,
    bf16_full_eval=True,
    load_best_model_at_end=True,
    greater_is_better=True,
    report_to="none",
    max_grad_norm=1.0,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    lr_scheduler_type="linear",
    metric_for_best_model="f1_macro",)

trainer = Trainer(model=model, args=training_args,
    train_dataset=train_dataset,eval_dataset=val_dataset,compute_metrics=compute_metrics,)
trainer.train()

Class weights: tensor([0.3124, 0.9980, 4.1201, 2.2871, 8.5345])


Some weights of MBertMulticlass were not initialized from the model checkpoint at /workspace/models/bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pos_weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Micro,F1 Weighted
1,1.110100,0.746650,0.723117,0.507083,0.723117,0.710825
2,0.789400,0.813127,0.689351,0.524532,0.689351,0.713234
3,0.602600,0.725082,0.745455,0.569009,0.745455,0.760860
4,0.497300,0.727579,0.766753,0.578454,0.766753,0.773527
5,0.284600,0.860044,0.772987,0.593435,0.772987,0.778184


Checkpoint destination directory /workspace/lc_models/mBERT/checkpoint-5415 already exists and is non-empty. Saving will proceed but saved results may be invalid.


TrainOutput(global_step=5415, training_loss=0.7150991213949103, metrics={'train_runtime': 821.4618, 'train_samples_per_second': 105.452, 'train_steps_per_second': 6.592, 'total_flos': 5698152272736000.0, 'train_loss': 0.7150991213949103, 'epoch': 5.0})

In [12]:
trainer.save_model(save_dir)
tokenizer.save_pretrained(save_dir)
print(f"✅ Model saved to {save_dir}")


config = BertConfig.from_pretrained(save_dir)
config.output_attentions = False
config.use_cache = False
model = MBertMulticlass.from_pretrained(save_dir, config=config, ignore_mismatched_sizes=True, use_safetensors=True)
tokenizer = AutoTokenizer.from_pretrained(save_dir)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

gc.collect()
torch.cuda.empty_cache()

✅ Model saved to /workspace/ft_models/t2-m1-mBERT


Some weights of MBertMulticlass were not initialized from the model checkpoint at /workspace/ft_models/t2-m1-mBERT and are newly initialized because the shapes did not match:
- pos_weight: found shape torch.Size([5]) in the checkpoint and torch.Size([]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [13]:
from torch.utils.data import DataLoader
CHUNK_SIZE = 256
test_loader = DataLoader(test_dataset, batch_size=CHUNK_SIZE,shuffle=False,num_workers=2,pin_memory=True)
all_logits = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        if logits.dim() == 2 and logits.size(1) == 1:
            logits = logits.squeeze(1)
        all_logits.append(logits.detach().float().cpu().numpy())
        del outputs, logits, input_ids, attention_mask
        gc.collect()
        torch.cuda.empty_cache()
all_logits = np.concatenate(all_logits, axis=0)
preds = np.argmax(all_logits, axis=-1)
labels = np.array(test_dataset["labels"])
report = classification_report( labels,preds,labels=range(5),
    target_names=['NON 0', 'CYB 1', 'HAT 2', 'PRO 3', 'OTH 4'],
    output_dict=True,zero_division=0,)
print(report)
gc.collect()
torch.cuda.empty_cache()

{'NON 0': {'precision': 0.9089968976215098, 'recall': 0.8558909444985394, 'f1-score': 0.8816449348044132, 'support': 3081.0}, 'CYB 1': {'precision': 0.7008712487899322, 'recall': 0.7502590673575129, 'f1-score': 0.7247247247247247, 'support': 965.0}, 'HAT 2': {'precision': 0.46488294314381273, 'recall': 0.594017094017094, 'f1-score': 0.5215759849906192, 'support': 234.0}, 'PRO 3': {'precision': 0.5726872246696035, 'recall': 0.6175771971496437, 'f1-score': 0.5942857142857143, 'support': 421.0}, 'OTH 4': {'precision': 0.40476190476190477, 'recall': 0.45535714285714285, 'f1-score': 0.42857142857142855, 'support': 112.0}, 'accuracy': 0.7918138375233742, 'macro avg': {'precision': 0.6104400437973526, 'recall': 0.6546202891759865, 'f1-score': 0.6301605574753799, 'support': 4813.0}, 'weighted avg': {'precision': 0.8045247164490017, 'recall': 0.7918138375233742, 'f1-score': 0.7969978121117489, 'support': 4813.0}}


In [15]:
def auprc(scores, y_true):
    from sklearn.metrics import average_precision_score
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    if len(y_true) == 0 or y_true.sum() == 0 or y_true.sum() == len(y_true):return 0.0
    return average_precision_score(y_true, scores)

def token_f1(y_pred, y_true):
    y_pred = np.asarray(y_pred)
    y_true = np.asarray(y_true)
    tp = np.sum((y_pred == 1) & (y_true == 1))
    fp = np.sum((y_pred == 1) & (y_true == 0))
    fn = np.sum((y_pred == 0) & (y_true == 1))
    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1 = 2 * precision * recall / (precision + recall + 1e-12)
    return f1

def iou(y_pred, y_true):
    pred_pos = np.where(y_pred == 1)[0]
    gold_pos = np.where(y_true == 1)[0]
    if len(pred_pos) == 0 or len(gold_pos) == 0:return 0.0
    intersection = len(set(pred_pos) & set(gold_pos))
    union = len(set(pred_pos) | set(gold_pos))
    return intersection / union if union > 0 else 0.0

def subword_to_word_importance(token_scores, word_ids):
    if len(token_scores) == 0: return np.array([])
    word_importance = []
    current_word_id = None
    current = 0 
    for i, wid in enumerate(word_ids):
        if wid is None:  continue
        if wid != current_word_id:
            if current_word_id is not None: word_importance.append(current)
            current_word_id = wid
            current = token_scores[i]
        else:
            current = current + token_scores[i] 
    if current_word_id is not None: word_importance.append(current)
    return np.array(word_importance)

def binarize_topk_nonzero(scores, k):
    scores = np.asarray(scores)
    nonzero_idx = np.where(scores > 0)[0]
    if len(nonzero_idx) == 0:
        return np.zeros(len(scores), dtype=int)
    k = min(k, len(nonzero_idx))
    top_k = nonzero_idx[np.argsort(scores[nonzero_idx])[-k:]]
    binary = np.zeros(len(scores), dtype=int)
    binary[top_k] = 1
    return binary

def mask_words(inputs, tokenizer, keep_word_mask, text):
    input_ids = inputs["input_ids"].clone()          # shape (1, seq_len)
    attention_mask = inputs["attention_mask"].clone()    
    word_ids = inputs.get("word_ids")
    if word_ids is None:
        tokenized = tokenizer(text, return_offsets_mapping=True, return_tensors="pt")
        word_ids = tokenized.word_ids()   
    mask_token_id = tokenizer.mask_token_id or tokenizer.unk_token_id    
    current_word  = -1    
    for pos in range(1, input_ids.size(1)):  
        if word_ids[pos] is None:continue
        if word_ids[pos] != current_word:
            current_word += 1
            if current_word >= len(keep_word_mask): break
            keep = keep_word_mask[current_word] 
        if not keep:input_ids[0, pos] = mask_token_id
    return {"input_ids": input_ids, "attention_mask": attention_mask}
    
def get_class_probability(model, inputs, class_idx, output_attentions=False):
    with torch.no_grad():
        model_inputs = {k: v for k, v in inputs.items() if k in ["input_ids", "attention_mask"]}
        outputs = model(**model_inputs, output_attentions=output_attentions)
        logits = outputs.logits 
        probs = torch.softmax(logits, dim=-1)
        prob = probs[0, class_idx].clamp(0.0, 1.0).item()
    return prob

def get_lime_importance(model, text, tokenizer, target_class, max_features=30, num_samples=2000):
    def predict_proba(texts):
        enc = tokenizer(texts,padding=True,truncation=True,
            max_length=MAX_LENGTH,return_tensors="pt",).to(device)
        with torch.no_grad():
            logits = model(**enc).logits  # (n_texts, 5)
            probs = torch.softmax(logits, dim=-1).cpu().numpy()
            probs = np.nan_to_num(probs, nan=0.0, posinf=1.0, neginf=0.0)
        return probs
    explainer = LimeTextExplainer( class_names=['NON 0', 'CYB 1', 'HAT 2', 'PRO 3', 'OTH 4'],
        split_expression=r"\s+",bow=False,kernel_width=25,random_state=42,)
    exp = explainer.explain_instance(
        text_instance=text,classifier_fn=predict_proba,
        num_features=max_features,num_samples=num_samples,
        labels=[target_class],)   
    lime_word_scores = dict(exp.as_list(label=target_class))
    words = text.split()
    word_importance = np.zeros(len(words), dtype=np.float32)
    for i, w in enumerate(words):
        if w in lime_word_scores:
            word_importance[i] = lime_word_scores[w]
    return word_importance

results_attn = {"auprc": [], "token_f1": [], "iou": [], "comprehensiveness": [], "sufficiency": []}
attn_miss = 0
REMOVE_CHARS = r"[|۔؟\?,\.،!_:\"\-_]"
for _, row in test_df[test_df[ycolumn] != 0].iterrows():  
    text = row[xcolumn]
    text = re.sub(REMOVE_CHARS, "", text)
    gold_word_rationale = np.array(row["rationale"])
    if len(gold_word_rationale) == 0: continue
    gold_label = int(row[ycolumn])  
    tokenized = tokenizer(text,truncation=True,
        padding="max_length",max_length=MAX_LENGTH,
        return_offsets_mapping=True,return_tensors="pt",)
    word_ids = tokenized.word_ids()
    inputs = {"input_ids": tokenized["input_ids"].to(device),
        "attention_mask": tokenized["attention_mask"].to(device),
        "word_ids": word_ids,}
    orig_prob = get_class_probability(model, inputs, gold_label, output_attentions=False)  
    with torch.no_grad(): 
        outputs = model(**inputs, output_attentions=True)
    attentions = outputs.attentions
    last_layer_attn = attentions[-1].squeeze(0)
    avg_last_attn = last_layer_attn.mean(dim=0)
    token_importance = avg_last_attn[0].cpu().numpy()
    token_importance = np.maximum(token_importance, 0.0)
    token_importance[0] = 0.0
    word_importance = subword_to_word_importance(token_importance, word_ids)
    # align lengths (same as original)
    while len(gold_word_rationale) > len(word_importance):
        zeros = np.where(gold_word_rationale == 0)[0]
        idx = zeros[0] if len(zeros) > 0 else np.where(gold_word_rationale == 1)[0][0]
        gold_word_rationale = np.delete(gold_word_rationale, idx)
    while len(gold_word_rationale) < len(word_importance):
        gold_word_rationale = np.append(gold_word_rationale, 0)

    if len(word_importance) != len(gold_word_rationale):
        print("Length mismatch — skipping")
        attn_miss += 1
        continue

    k = min(8, len(word_importance))
    pred_word_binary = binarize_topk_nonzero(word_importance, k=k)

    results_attn["auprc"].append(auprc(word_importance, gold_word_rationale))
    results_attn["token_f1"].append(token_f1(pred_word_binary, gold_word_rationale))
    results_attn["iou"].append(iou(pred_word_binary, gold_word_rationale))

    # Faithfulness (now using gold class probability)
    comp_keep_mask = pred_word_binary == 0
    masked_comp_inputs = mask_words(inputs, tokenizer, comp_keep_mask, text)
    comp_prob = get_class_probability(model, masked_comp_inputs, gold_label, output_attentions=False)
    results_attn["comprehensiveness"].append(orig_prob - comp_prob)

    suff_keep_mask = pred_word_binary == 1
    masked_suff_inputs = mask_words(inputs, tokenizer, suff_keep_mask, text)
    suff_prob = get_class_probability(model, masked_suff_inputs, gold_label, output_attentions=False)
    results_attn["sufficiency"].append(orig_prob - suff_prob)

print("\nDataset-level results (macro average over positive examples):\n")
print("attn_miss:",attn_miss)
for name, values in results_attn.items():
    if not values:
        print(f"{name:18}: no valid examples")
        continue
    mean = np.mean(values)
    std  = np.std(values)
    n    = len(values)
    print(f"{name:18}: {mean:.4f} ± {std:.4f}  (n={n})")


Dataset-level results (macro average over positive examples):

attn_miss: 0
auprc             : 0.5404 ± 0.2719  (n=1732)
token_f1          : 0.4138 ± 0.2235  (n=1732)
iou               : 0.2883 ± 0.1985  (n=1732)
comprehensiveness : 0.5730 ± 0.4644  (n=1732)
sufficiency       : 0.3054 ± 0.4198  (n=1732)


In [16]:
results_lime = {"auprc": [], "token_f1": [], "iou": [], "comprehensiveness": [], "sufficiency": []}
i = 0
SAVE_result = "/workspace/Results/t2-t1-results_lime.csv"
for idx, row in test_df[test_df[ycolumn] != 0].iterrows():
    if i % 100 == 0:
        print(f"Processed {i}, saving full results...")
        pd.DataFrame(results_lime).to_csv(SAVE_result,index=False)
    i +=1
    text = row[xcolumn]
    gold_word_rationale = np.array(row["rationale"])
    if len(gold_word_rationale) == 0:continue
    gold_label = int(row[ycolumn])
    tokenized = tokenizer(text,truncation=True,
        padding="max_length",max_length=MAX_LENGTH,
        return_offsets_mapping=True,return_tensors="pt",)
    word_ids = tokenized.word_ids()
    inputs = {"input_ids": tokenized["input_ids"].to(device),
        "attention_mask": tokenized["attention_mask"].to(device),
        "word_ids": word_ids,}
    lime_scores = get_lime_importance(model=model,
        text=text,tokenizer=tokenizer,target_class=gold_label, 
        max_features=len(gold_word_rationale),num_samples=1000,)

    word_importance = np.maximum(lime_scores, 0.0)
    word_importance = word_importance / (word_importance.sum() + 1e-12)
    orig_prob = get_class_probability(model, inputs, gold_label)
    if len(word_importance) != len(gold_word_rationale):
        print(f"Length mismatch (example {idx}) — skipping")
        continue
    k = min(8, len(word_importance))
    pred_word_binary = binarize_topk_nonzero(word_importance, k=k)
    results_lime["auprc"].append(auprc(word_importance, gold_word_rationale))
    results_lime["token_f1"].append(token_f1(pred_word_binary, gold_word_rationale))
    results_lime["iou"].append(iou(pred_word_binary, gold_word_rationale))
    comp_keep_mask = pred_word_binary == 0
    masked_comp_inputs = mask_words(inputs, tokenizer, comp_keep_mask, text)
    comp_prob = get_class_probability(model, masked_comp_inputs, gold_label, output_attentions=False)
    results_lime["comprehensiveness"].append(orig_prob - comp_prob)
    suff_keep_mask = pred_word_binary == 1
    masked_suff_inputs = mask_words(inputs, tokenizer, suff_keep_mask, text)
    suff_prob = get_class_probability(model, masked_suff_inputs, gold_label, output_attentions=False)
    results_lime["sufficiency"].append(orig_prob - suff_prob)

pd.DataFrame(results_lime).to_csv(SAVE_result,index=False) 

Processed 0, saving full results...
Processed 100, saving full results...
Processed 200, saving full results...
Processed 300, saving full results...
Processed 400, saving full results...
Processed 500, saving full results...
Processed 600, saving full results...
Processed 700, saving full results...
Processed 800, saving full results...
Processed 900, saving full results...
Processed 1000, saving full results...
Processed 1100, saving full results...
Processed 1200, saving full results...
Processed 1300, saving full results...
Processed 1400, saving full results...
Processed 1500, saving full results...
Processed 1600, saving full results...
Processed 1700, saving full results...


In [19]:
def r2(x):
    if x == "" or x is None:return ""
    return round(float(x), 2)
    
script_name = os.path.basename(__file__) if "__file__" in globals() else "interactive"
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
file_exists = os.path.isfile(result_path)
modelname = 'T1-M1-mBert-Attn-Lime'
with open(result_path, mode="a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["######################################################################"])
    writer.writerow([modelname, script_name, timestamp,])
    writer.writerow(["Accuracy", r2(report["accuracy"]) ])
    writer.writerow(["Class", "Precision", "Recall", "F1-Score", "Support"])
    for class_name, metrics in report.items():
        if isinstance(metrics, dict):writer.writerow([class_name,r2(metrics["precision"]), r2(metrics["recall"]), r2(metrics["f1-score"]),r2(metrics["support"])])
    for metric_name, values in results_attn.items():
        mean_val = np.mean(values)
        std_val = np.std(values)
        writer.writerow([metric_name,"-ATTN--", r2(mean_val),  r2(std_val) ])
    for metric_name, values in results_lime.items():
        mean_val = np.mean(values)
        std_val = np.std(values)
        writer.writerow([metric_name,"-LIME--", r2(mean_val),  r2(std_val) ])


script_name = os.path.basename(__file__) if "__file__" in globals() else "interactive"
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
modelname = 'T1-M1-mBert-Attn-Lime'
print("######################################################################")
print(f"Model: {modelname} | Script: {script_name} | Timestamp: {timestamp}\n")
print('Accuracy:',r2(report["accuracy"]))
print(f"{'Class':<15} {'Precision':<10} {'Recall':<10} {'F1-Score':<10} {'Support':<10}")
for class_name, metrics in report.items():
    if isinstance(metrics, dict):
        print(f"{class_name:<15} {r2(metrics['precision']):<10} {r2(metrics['recall']):<10} "
              f"{r2(metrics['f1-score']):<10} {r2(metrics['support']):<10}")
# Token-level / Faithfulness metrics
print("\nToken-level / Faithfulness Metrics (Mean ± Std):")
print(f"{'Metric':<20} {'Attention':<15} {'LIME':<15}")
for metric_name in results_attn.keys():
    attn_mean = r2(np.mean(results_attn[metric_name]))
    attn_std = r2(np.std(results_attn[metric_name]))
    lime_mean = r2(np.mean(results_lime[metric_name]))
    lime_std = r2(np.std(results_lime[metric_name]))
    print(f"{metric_name:<20} {attn_mean} ± {attn_std:<11} {lime_mean} ± {lime_std:<11}")

   

######################################################################
Model: T1-M1-mBert-Attn-Lime | Script: interactive | Timestamp: 2026-02-28 11:38:51

Accuracy: 0.79
Class           Precision  Recall     F1-Score   Support   
NON 0           0.91       0.86       0.88       3081.0    
CYB 1           0.7        0.75       0.72       965.0     
HAT 2           0.46       0.59       0.52       234.0     
PRO 3           0.57       0.62       0.59       421.0     
OTH 4           0.4        0.46       0.43       112.0     
macro avg       0.61       0.65       0.63       4813.0    
weighted avg    0.8        0.79       0.8        4813.0    

Token-level / Faithfulness Metrics (Mean ± Std):
Metric               Attention       LIME           
auprc                0.54 ± 0.27        0.61 ± 0.27       
token_f1             0.41 ± 0.22        0.42 ± 0.2        
iou                  0.29 ± 0.2         0.29 ± 0.18       
comprehensiveness    0.57 ± 0.46        0.55 ± 0.44       
sufficienc